# PDF to Editable DOCX Converter (Fully Self‑Sufficient)
This notebook converts PDF files into fully editable Word documents (DOCX) with high layout accuracy.

- Uses **pdf2docx** for text‑based PDFs
- Automatically detects scanned (image‑only) PDFs and applies **OCRmyPDF** to add a searchable text layer before conversion
- Preserves text positions, tables, and images
- Supports single files, whole folders, specific page ranges, and forced OCR

**No manual setup required** – the first cell handles all dependencies transparently.

In [14]:
import sys, subprocess, venv, os, tempfile, shutil, site, logging
from pathlib import Path
from typing import List, Optional

# ----------------------------------------------------------------------
# Robust import helper – creates a temp venv and installs packages if needed
def _ensure_imports():
    packages = {'fitz': 'PyMuPDF', 'pdf2docx': 'pdf2docx'}
    missing = []
    for mod, pkg in packages.items():
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)

    if missing:
        print(f"Missing packages: {missing}. Setting up temporary environment…")
        # Create a temporary venv
        tmp_venv = tempfile.mkdtemp(prefix='pdfconv_venv')
        venv.create(tmp_venv, with_pip=True)

        # Determine Python executable inside the venv
        if os.name == 'nt':
            python_exe = os.path.join(tmp_venv, 'Scripts', 'python.exe')
        else:
            python_exe = os.path.join(tmp_venv, 'bin', 'python')

        # Install packages
        pip_install = [python_exe, '-m', 'pip', 'install', '--quiet'] + missing
        subprocess.check_call(pip_install)

        # Add the venv's site‑packages to sys.path so this kernel can use them
        # Use site.getusersitepackages() logic, but for the venv we locate it manually
        if os.name == 'nt':
            site_dir = os.path.join(tmp_venv, 'Lib', 'site-packages')
        else:
            # Get the lib directory name (e.g., python3.10)
            lib_dir = 'python' + sys.version[:3]
            site_dir = os.path.join(tmp_venv, 'lib', lib_dir, 'site-packages')

        if os.path.isdir(site_dir):
            sys.path.insert(0, site_dir)
            # Also add site-packages from base to be safe (for dependencies)
            try:
                out = subprocess.check_output([python_exe, '-c', 'import site; print(site.getsitepackages()[0])'])
                venv_site = out.decode().strip()
                if venv_site not in sys.path:
                    sys.path.insert(0, venv_site)
            except Exception:
                pass
        else:
            raise RuntimeError(f"Could not find site‑packages in {tmp_venv}")

        print("Temporary environment ready. Importing packages…")
        # Try imports again
        for mod in packages:
            try:
                __import__(mod)
            except ImportError as e:
                raise ImportError(f"Failed to import {mod} even after installation: {e}")

        print("All dependencies successfully loaded.\n")

_ensure_imports()

# Now we can import safely
import fitz  # PyMuPDF
from pdf2docx import Converter

# Optional OCR support
try:
    import ocrmypdf
    _has_ocr = True
except ImportError:
    _has_ocr = False
    ocrmypdf = None
    print("Note: 'ocrmypdf' is not installed – OCR for scanned PDFs will be unavailable.\n")

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

Note: 'ocrmypdf' is not installed – OCR for scanned PDFs will be unavailable.



In [15]:
def is_scanned(pdf_path: str, text_threshold: int = 100) -> bool:
    """
    Returns True if the PDF contains very little text (likely scanned).
    """
    try:
        doc = fitz.open(pdf_path)
        total_chars = sum(len(page.get_text()) for page in doc)
        doc.close()
        return total_chars < text_threshold
    except Exception as e:
        logger.warning(f"Could not analyze {pdf_path}: {e}")
        return False

In [16]:
def ocr_pdf(input_pdf: str, output_pdf: str) -> bool:
    """
    Run OCRmyPDF to add a text layer to the PDF.
    Returns True on success.
    """
    if ocrmypdf is None:
        logger.error("ocrmypdf is not installed. OCR not possible.")
        return False
    try:
        logger.info(f"Running OCR on: {input_pdf}")
        ocrmypdf.ocr(input_pdf, output_pdf, deskew=True, clean=True, skip_text=True)
        logger.info("OCR completed successfully.")
        return True
    except Exception as e:
        logger.error(f"OCR failed: {e}")
        return False

In [17]:
def convert_pdf_to_docx(
    pdf_path: str,
    docx_path: str,
    start_page: int = 0,
    end_page: Optional[int] = None,
    force_ocr: bool = False,
    pages: Optional[List[int]] = None
) -> bool:
    """
    Convert a single PDF to DOCX, with optional OCR for scanned files.
    """
    pdf_to_use = pdf_path
    tmp_pdf = None

    if force_ocr or is_scanned(pdf_path):
        if force_ocr:
            logger.info("OCR forced by user.")
        else:
            logger.info("PDF seems scanned (very little text). Applying OCR...")

        tmp_fd, tmp_pdf = tempfile.mkstemp(suffix=".pdf")
        os.close(tmp_fd)
        if not ocr_pdf(pdf_path, tmp_pdf):
            logger.error("OCR failed; falling back to original PDF (may be empty).")
            if tmp_pdf and os.path.exists(tmp_pdf):
                os.unlink(tmp_pdf)
                tmp_pdf = None
        else:
            pdf_to_use = tmp_pdf

    try:
        logger.info(f"Converting: {pdf_to_use} -> {docx_path}")
        cv = Converter(pdf_to_use)
        if pages is not None:
            cv.convert(docx_path, pages=pages)
        else:
            cv.convert(docx_path, start=start_page, end=end_page)
        cv.close()
        logger.info(f"Successfully created: {docx_path}")
        return True
    except Exception as e:
        logger.error(f"Conversion failed for {pdf_path}: {e}")
        return False
    finally:
        if tmp_pdf and os.path.exists(tmp_pdf):
            os.unlink(tmp_pdf)

In [18]:
def convert_pdfs(
    input_path: str,
    output_dir: Optional[str] = None,
    start_page: int = 0,
    end_page: Optional[int] = None,
    pages: Optional[List[int]] = None,
    force_ocr: bool = False,
    no_ocr: bool = False
) -> List[str]:
    """
    Convert one or more PDFs to DOCX.
    - input_path: single PDF file, or a folder containing PDFs.
    - output_dir: where to save the DOCX files (default: same as input).
    Returns list of successfully created DOCX paths.
    """
    p = Path(input_path)
    pdf_paths = []
    if p.is_dir():
        pdf_paths = list(p.glob("*.pdf"))
    elif p.is_file() and p.suffix.lower() == ".pdf":
        pdf_paths.append(p)
    else:
        raise ValueError(f"Invalid input: {input_path} – must be a PDF file or directory.")

    if not pdf_paths:
        logger.warning("No PDF files found.")
        return []

    if output_dir:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
    else:
        out_dir = None

    success_files = []
    for pdf_file in pdf_paths:
        if out_dir:
            docx_path = out_dir / (pdf_file.stem + ".docx")
        else:
            docx_path = pdf_file.with_suffix(".docx")

        ocr_flag = force_ocr and not no_ocr
        if no_ocr:
            ocr_flag = False

        if convert_pdf_to_docx(
            str(pdf_file),
            str(docx_path),
            start_page=start_page,
            end_page=end_page,
            force_ocr=ocr_flag,
            pages=pages
        ):
            success_files.append(str(docx_path))

    logger.info(f"Done. {len(success_files)}/{len(pdf_paths)} file(s) converted.")
    return success_files

## Usage Examples
Modify the paths and run the cells below to convert your PDFs.

In [19]:
# Example 1: Convert a single PDF (text-based)
converted = convert_pdfs("document.pdf")
print("Created:", converted)

[INFO] Converting: document.pdf -> document.docx
[INFO] Start to convert document.pdf
[INFO] [1/4] Opening document...
[INFO] [2/4] Analyzing document...
[INFO] [3/4] Parsing pages...
[INFO] (1/1) Page 1
[INFO] [4/4] Creating pages...
[INFO] (1/1) Page 1
[INFO] Terminated in 0.25s.
[INFO] Successfully created: document.docx
[INFO] Done. 1/1 file(s) converted.


Created: ['document.docx']


In [20]:
# Example 2: Convert all PDFs in a folder to a separate output folder
#converted = convert_pdfs("input_pdfs/", output_dir="output_docx/")
#print("Created:", converted)

In [21]:
# Example 3: Convert only pages 2, 3, and 5 (0-indexed)
#converted = convert_pdfs("report.pdf", pages=[1, 2, 4])
#print("Created:", converted)

In [22]:
# Example 4: Force OCR on a scanned PDF
#converted = convert_pdfs("scanned_image.pdf", force_ocr=True)
#print("Created:", converted)

In [23]:
# Example 5: Disable automatic OCR (keep scanned PDF as-is)
#converted = convert_pdfs("scanned_image.pdf", no_ocr=True)
#print("Created:", converted)

## Notes
- The output DOCX will be placed next to the input PDF unless `output_dir` is specified.
- For scanned PDFs, `ocrmypdf` and Tesseract need to be installed separately (see https://github.com/tesseract-ocr/tesseract).
- Layout fidelity is excellent, but complex magazine‑style columns may need minor manual adjustment.
- Fonts are substituted if the originals aren’t installed on your system.

Enjoy your fully editable Word documents!